# Vehicle Sales Business Analytics

Python analysis of vehicle sales using Pandas, NumPy, Matplotlib and Seaborn.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 1. Load Dataset


In [ ]:
file_path = '../data/car_prices_sample.csv'
df = pd.read_csv(file_path)
print('Rows:', f'{len(df):,}')
print('Columns:', len(df.columns))
df.head()


## 2. Data Quality Check


In [ ]:
print(df.info())
print('\nDuplicates:', df.duplicated().sum())
missing = df.isna().sum().sort_values(ascending=False)
print('\nMissing values:')
display(missing[missing > 0].to_frame('missing_values'))


## 3. Data Cleaning


In [ ]:
clean = df.copy()
text_cols = ['make','model','trim','body','transmission','state','color','interior','seller']
for col in text_cols:
    clean[col] = clean[col].astype('string').str.strip()
for col in ['year','condition','odometer','mmr','sellingprice']:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
clean['saledate'] = pd.to_datetime(clean['saledate'], errors='coerce')
clean = clean.drop_duplicates().copy()
clean = clean[(clean['sellingprice'].notna()) & (clean['sellingprice'] >= 0)]
clean = clean[(clean['odometer'].isna()) | (clean['odometer'] >= 0)]
print('Clean rows:', f'{len(clean):,}')


## 4. Feature Engineering


In [ ]:
clean['price_difference'] = clean['sellingprice'] - clean['mmr']
clean['price_difference_pct'] = np.where(clean['mmr'] > 0, clean['price_difference'] / clean['mmr'] * 100, np.nan)
clean['vehicle_age'] = np.where(clean['year'].notna(), 2015 - clean['year'], np.nan)
clean['mileage_category'] = pd.cut(clean['odometer'], [-np.inf,30000,70000,100000,np.inf], labels=['Low Mileage','Medium Mileage','High Mileage','Very High Mileage'])
clean['price_range'] = pd.cut(clean['sellingprice'], [-np.inf,10000,20000,30000,50000,np.inf], labels=['Under $10K','$10K-$20K','$20K-$30K','$30K-$50K','$50K+'])
clean.head()


## 5. Core KPIs


In [ ]:
kpis = pd.Series({'Total Vehicles': len(clean), 'Total Sales': clean['sellingprice'].sum(), 'Average Selling Price': clean['sellingprice'].mean(), 'Average Market Value': clean['mmr'].mean(), 'Average Mileage': clean['odometer'].mean(), 'Average Price Difference': clean['price_difference'].mean()})
kpis


## 6. Top Brands


In [ ]:
top_brands = (clean.dropna(subset=['make']).groupby('make').agg(vehicles_sold=('make','size'), total_sales=('sellingprice','sum'), avg_selling_price=('sellingprice','mean')).sort_values('vehicles_sold', ascending=False).head(10))
top_brands


In [ ]:
plt.figure(figsize=(10,5))
top_brands['vehicles_sold'].sort_values().plot(kind='barh')
plt.title('Top Vehicle Brands by Sales Volume')
plt.xlabel('Vehicles Sold')
plt.ylabel('Brand')
plt.tight_layout()
plt.show()


## 7. Regional Analysis


In [ ]:
state_analysis = (clean.dropna(subset=['state']).groupby('state').agg(vehicles_sold=('state','size'), total_sales=('sellingprice','sum'), avg_selling_price=('sellingprice','mean')).sort_values('total_sales', ascending=False))
state_analysis.head(10)


In [ ]:
top_states = state_analysis.head(10).sort_values('total_sales')
plt.figure(figsize=(10,5))
top_states['total_sales'].plot(kind='barh')
plt.title('Top States by Vehicle Sales Revenue')
plt.xlabel('Total Sales')
plt.ylabel('State')
plt.tight_layout()
plt.show()


## 8. Mileage vs Selling Price


In [ ]:
mileage_analysis = (clean.dropna(subset=['mileage_category']).groupby('mileage_category', observed=False).agg(vehicles_sold=('sellingprice','size'), avg_selling_price=('sellingprice','mean'), avg_market_value=('mmr','mean')))
mileage_analysis


In [ ]:
plot_df = clean.dropna(subset=['odometer','sellingprice']).sample(min(5000, len(clean.dropna(subset=['odometer','sellingprice']))), random_state=42)
plt.figure(figsize=(10,5))
plt.scatter(plot_df['odometer'], plot_df['sellingprice'], alpha=0.25)
plt.title('Mileage vs Selling Price')
plt.xlabel('Mileage')
plt.ylabel('Selling Price')
plt.tight_layout()
plt.show()


## 9. Market Value vs Selling Price


In [ ]:
brand_pricing = (clean.dropna(subset=['make','mmr','sellingprice']).groupby('make').agg(vehicles_sold=('make','size'), avg_market_value=('mmr','mean'), avg_selling_price=('sellingprice','mean'), avg_price_difference=('price_difference','mean')).query('vehicles_sold >= 50').sort_values('avg_price_difference', ascending=False).head(15))
brand_pricing


## 10. Top Models and Price Ranges


In [ ]:
top_models = (clean.dropna(subset=['make','model']).groupby(['make','model']).agg(vehicles_sold=('model','size'), avg_selling_price=('sellingprice','mean')).sort_values('vehicles_sold', ascending=False).head(10))
print('Top Models')
display(top_models)
print('\nPrice Ranges')
display(clean.groupby('price_range', observed=False).agg(vehicles_sold=('sellingprice','size'), avg_selling_price=('sellingprice','mean')))


## 11. Condition Analysis


In [ ]:
condition_analysis = (clean.dropna(subset=['condition']).groupby('condition').agg(vehicles_sold=('condition','size'), avg_selling_price=('sellingprice','mean'), avg_market_value=('mmr','mean')).sort_index())
condition_analysis


## 12. Business Takeaways


In [ ]:
top_brand = top_brands.index[0] if len(top_brands) else 'N/A'
top_state = state_analysis.index[0] if len(state_analysis) else 'N/A'
best_mileage = mileage_analysis['avg_selling_price'].idxmax() if len(mileage_analysis) else 'N/A'
print('Business Takeaways')
print('1. Highest sales-volume brand in the sample:', top_brand)
print('2. Highest-revenue state in the sample:', top_state)
print('3. Mileage group with highest average selling price:', best_mileage)
print('4. Compare selling price with MMR to identify pricing opportunities.')
print('5. Use brand, region, mileage and condition together when evaluating vehicle performance.')
